## Extract data

In [891]:
import pandas as pd
import numpy as np

In [ ]:
# # Ensure the required library is installed 
# (Can we do it (openpyxl)? Will it be a problem?)
# %pip install openpyxl

# Read the Excel file
file_path = "bitre_fatalities_dec2024.xlsx"
df = pd.read_excel(file_path, sheet_name="BITRE_Fatality", skiprows=4) # Can be improved


In [893]:
# Clean the columnnames
# Remove leading and trailing whitespace, convert to lowercase, and replace spaces with underscores

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df.columns

# change sa4_name_2021 to sa4_name, national_lga_name_2021 to lga_name
df.rename(columns={
    'national_lga_name_2021': 'lga_name',
    'national_road_type': 'road_type'
}, inplace=True)
df.columns


Index(['crash_id', 'state', 'month', 'year', 'dayweek', 'time', 'crash_type',
       'bus_involvement', 'heavy_rigid_truck_involvement',
       'articulated_truck_involvement', 'speed_limit', 'road_user', 'gender',
       'age', 'national_remoteness_areas', 'sa4_name_2021', 'lga_name',
       'road_type', 'christmas_period', 'easter_period', 'age_group',
       'day_of_week', 'time_of_day'],
      dtype='object')

In [894]:
# Add a serial number for each person killed in the accident
df['victim_number'] = df.groupby('crash_id').cumcount() + 1

# move the victim_number column to the front
cols = df.columns.tolist()
cols.insert(1, cols.pop(cols.index('victim_number')))
df = df[cols]

In [895]:
# drop 'age', 'road_user', 'gender', 'national_remoteness_areas', 'sa4_name_2021', 'day_of_week'
df.drop(columns=['age', 'road_user', 'gender', 'national_remoteness_areas', 'sa4_name_2021', 'day_of_week'], inplace=True)

## Data transformation

Data transformation to apply:

1. dayweek: Drop
2. time: Categorise into rush time and usual time:
    Rush hours: 
    Morning Peak:
    Typically between 7 am and 9 am, as commuters head to work or school. 

    Evening Peak:
    Typically between 4 pm and 6 pm, as commuters travel home from work or school. 

    Not holiday, not weekend

Can be improved according to the state, city and so on

3. bus_involvement, heavy_rigid_truck_involvement, articulated_truck_involvement - treat -9 missing values
4. speed_limit: Categorise as follows:
    For all except NT:
        0-40 - low
        41-50 - med
        51-80 - high
        81 - inf - very high
    
    For NT:
        0-40 - low
        41-60 - med
        61-80 - high
        81 - inf - very high

    treat -9 as missing value
5. road_user:
    treat Other/-9, Unknown - as missing value

6. gender:
    treat -9 - as missing value

7. age: drop

8. national_remoteness_areas:
    treat Unknown - as missing value

9. sa4_name_2021:
    treat Unknown, Blank - as missing value

10. national_lga_name_2021:
    treat Unknown, Blank - as missing value

11. national_road_type:
    treat Undetermined - as missing value

12. christmas_period, easter_period:
    transform into is_holiday

13. age_group:
    treat -9 - as missing value

14. day_of_week:
    treat Unknown - as missing value

15. time_of_day:
    treat Unknown - as missing value

### Categorizing variables

In [896]:
# convert the 'time' column to datetime format
df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S', errors='coerce').dt.time

# categorize time of day by rush hous:
# For all holiday == "No", day_of_week == "Weekday" set rush: 
# 07:00:00 - 09:00:00 = "Rush"
# 16:00:00 - 18:00:00 = "Rush"
Weekday = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

conditions = [
    (df['christmas_period'] == "No") & (df['easter_period'] == "No") & (df['dayweek'].isin(Weekday)) & (
    ((df['time'] >= pd.to_datetime("07:00:00", format='%H:%M:%S').time()) & (df['time'] <= pd.to_datetime("09:00:00", format='%H:%M:%S').time())) |
    ((df['time'] >= pd.to_datetime("16:00:00", format='%H:%M:%S').time()) & (df['time'] <= pd.to_datetime("18:00:00", format='%H:%M:%S').time()))
    ),
    (df['time'].isna())
]

choices = ["Rush", np.nan]

df['time_cat'] = np.select(conditions, choices, default="Not Rush")

# change 'nan' value to np.nan
df['time_cat'] = df['time_cat'].replace('nan', np.nan)

print(df['time_cat'].unique())


['Not Rush' 'Rush' nan]


In [897]:
# set  NaN values for all 'Other/-9', '-9', 'Unknown', 'Undetermined' in all columns
nan_values = ['Other/-9', '-9', 'Unknown', 'Undetermined', -9]
df.replace(nan_values, np.nan, inplace=True)


In [898]:
# check for right data type for speed_limit
try :
    df['speed_limit'] = df['speed_limit'].astype(float)
except ValueError:
    # If conversion to int fails, print values that cannot be converted
    print("Values that cannot be converted to int:")
    print(df[~df['speed_limit'].apply(lambda x: isinstance(x, int) or pd.isna(x))]['speed_limit'].unique())



Values that cannot be converted to int:
['<40']


<40 means less than 40, which is 'low' speed category
just set this value to 40

In [899]:
# set '<40' speed_limit to 40
df['speed_limit'] = df['speed_limit'].replace('<40', 40)

In [900]:
# speed_limit: Categorise as follows:
#     For all except NT:
#         0-40 - low
#         41-50 - med
#         51-80 - high
#         81 - inf - very high
    
#     For NT:
#         0-40 - low
#         41-60 - med
#         61-80 - high
#         81 - inf - very high

df['speed_limit'] = np.where(
    df['state'] != "NT",
    np.select(
        [
            (df['speed_limit'] > 0) & (df['speed_limit'] <= 40),
            (df['speed_limit'] >= 41) & (df['speed_limit'] <= 50),
            (df['speed_limit'] >= 51) & (df['speed_limit'] <= 80),
            (df['speed_limit'] > 80)
        ],
        ['Low', 'Med', 'High', 'Very High'],
        default=np.nan
    ),
    np.select(
        [
            (df['speed_limit'] > 0) & (df['speed_limit'] <= 40),
            (df['speed_limit'] >= 41) & (df['speed_limit'] <= 60),
            (df['speed_limit'] >= 61) & (df['speed_limit'] <= 80),
            (df['speed_limit'] > 80)
        ],
        ['Low', 'Med', 'High', 'Very High'],
        default=np.nan
    )
)

# change 'nan' value to np.nan
df['speed_limit'] = df['speed_limit'].replace('nan', np.nan)            # КОСТЫЛЬ
# check if there are any NaN values in speed_limit
print(df['speed_limit'].isna().sum())


1485


In [901]:
print(df['speed_limit'].unique())

['Very High' 'High' 'Med' nan 'Low']


In [902]:
# create holiday column with values: Christmas, Easter, NoHoliday
conditions = [
    (df['christmas_period'] == "Yes"),
    (df['easter_period'] == "Yes"),
    (df['christmas_period'] == "No") & (df['easter_period'] == "No")
]
choices = ["Christmas", "Easter", "NoHoliday"]
df['holiday'] = np.select(conditions, choices, default=np.nan)

In [903]:
# create vehicle_type column with values: bus, heavy_truck, articulated_truck, noHeavyVehicle
conditions = [
    (df['bus_involvement'] == "Yes") | (df['heavy_rigid_truck_involvement'] == "Yes") | (df['articulated_truck_involvement'] == "Yes"),
    (df['bus_involvement'] == "No") & (df['heavy_rigid_truck_involvement'] == "No") & (df['articulated_truck_involvement'] == "No")
]

choices = ["Heavy Vehicle Involved", "No Heavy Vehicle Involved"]

df['vehicle_type_involved'] = np.select(conditions, choices, default=np.nan)
df['vehicle_type_involved'] = df['vehicle_type_involved'].replace('nan', np.nan)            # КОСТЫЛЬ

## Population table

In [904]:
file_path = "Population.xlsx"
population = pd.read_excel(file_path, sheet_name="Table 1", skiprows=5) # Can be improved
print(population.head())

  Unnamed: 0             Unnamed: 1   2001   2002   2003   2004   2005   2006  \
0   LGA code  Local Government Area    no.    no.    no.    no.    no.    no.   
1      10050                 Albury  45265  45816  46180  46505  47004  47566   
2      10180               Armidale  27906  27774  27610  27410  27350  27377   
3      10250                Ballina  37856  38417  38870  39120  39305  39537   
4      10300              Balranald   2751   2703   2661   2596   2545   2507   

    2007   2008  ...   2014   2015   2016   2017   2018   2019   2020   2021  \
0    no.    no.  ...    no.    no.    no.    no.    no.    no.    no.    no.   
1  48140  48518  ...  50990  51486  52171  53056  53922  54657  55466  56067   
2  27468  27788  ...  29015  29160  29310  29519  29631  29701  29600  29332   
3  39824  40020  ...  41881  42336  42993  43652  44385  44997  45663  46196   
4   2473   2433  ...   2376   2364   2330   2338   2308   2287   2257   2208   

    2022   2023  
0    no.    no

In [905]:
# Clean the columnnames
year_colnames = population.columns[2:].tolist()  # Get the first row for year column names
lga_colnames = population.iloc[0, :2].tolist()  # Get the first two columns for LGA names

# change 'local_government_area' to 'lga_name'
lga_colnames[1] = 'lga_name'

for i in range(len(lga_colnames)):
    lga_colnames[i] = lga_colnames[i].strip().lower().replace(' ', '_').replace('/', '_')
population.columns = lga_colnames + year_colnames  # Combine the two lists
population = population[1:-2].reset_index(drop=True)  # Skip the first row and the last 2 rows with '© Commonwealth of Australia' and Total Australia
population.head()
population.tail()

,lga_code,lga_name,2001,2002,2003,2004,2005,2006,2007,2008,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
542,74660,West Arnhem,6241,6198,6161,6182,6304,6452,6505,6679,...,7157,7060,6941,6943,6985,7059,7130,7182,7258,7407
543,74680,West Daly,2543,2612,2674,2752,2864,2986,3023,3151,...,3587,3598,3601,3536,3476,3427,3422,3422,3434,3426
544,79399,Unincorporated NT,7027,7072,7058,7128,7447,7664,7879,7995,...,7970,7112,7065,7034,7023,7102,7263,7420,7571,7713
545,89399,Unincorporated ACT,321538,324627,327357,328940,331399,335170,342644,348368,...,388799,395813,403104,415046,426081,435730,444903,452508,456915,466566
546,99399,Unincorp. Other Territories,542,464,441,428,413,386,370,370,...,361,367,2159,2243,2324,2382,2437,2530,2518,2516


In [906]:
# Transform population into long format
population_long = population.melt(
    id_vars = ["lga_code", "lga_name"],
    var_name = "year",
    value_name = "population"
)
population_long

,lga_code,lga_name,year,population
0,10050,Albury,2001,45265
1,10180,Armidale,2001,27906
2,10250,Ballina,2001,37856
3,10300,Balranald,2001,2751
4,10470,Bathurst,2001,35504
...,...,...,...,...
12576,74660,West Arnhem,2023,7407
12577,74680,West Daly,2023,3426
12578,79399,Unincorporated NT,2023,7713
12579,89399,Unincorporated ACT,2023,466566


In [907]:
# save population_long to excel
output_file_path = "population_long.xlsx"
population_long.to_excel(output_file_path, index=False)
print(f"Population data saved to {output_file_path}")

Population data saved to population_long.xlsx


## Designing dimention tables

In [908]:
print(df.columns)

Index(['crash_id', 'victim_number', 'state', 'month', 'year', 'dayweek',
       'time', 'crash_type', 'bus_involvement',
       'heavy_rigid_truck_involvement', 'articulated_truck_involvement',
       'speed_limit', 'lga_name', 'road_type', 'christmas_period',
       'easter_period', 'age_group', 'time_of_day', 'time_cat', 'holiday',
       'vehicle_type_involved'],
      dtype='object')


In [909]:
# create lga_code column in df by merging with population_long
df = df.merge(population_long[['lga_name', 'lga_code']].drop_duplicates(), on='lga_name', how='left')

def match_by_first_word(row):
    # Only try to match if lga_code is missing and lga_name exists
    if pd.isna(row['lga_code']) and pd.notna(row['lga_name']):
        first_word = row['lga_name'].split()[0]
        # Try to find match in population_long
        match = population_long[population_long['lga_name'].str.contains(rf'\b{first_word}\b', case=False, na=False)]
        if not match.empty:
            # If a match is found, update both lga_code and lga_name
            row['lga_code'] = match['lga_code'].values[0]
            row['lga_name'] = match['lga_name'].values[0]
    return row

# Apply the function row-wise
df = df.apply(match_by_first_word, axis=1)
# make lga_code to int, ignore NaN
df['lga_code'] = df['lga_code'].astype(pd.Int64Dtype())


In [910]:
# merge population_long with df on lga_code adding state column
location_dim = population_long[['lga_code', 'lga_name']].drop_duplicates()
location_dim = location_dim.merge(df[['lga_code', 'state']].drop_duplicates(), on='lga_code', how='left')
location_dim = location_dim[['lga_code', 'state', 'lga_name']]
location_dim

,lga_code,state,lga_name
0,10050,NSW,Albury
1,10180,NSW,Armidale
2,10250,NSW,Ballina
3,10300,NSW,Balranald
4,10470,NSW,Bathurst
...,...,...,...
542,74660,NT,West Arnhem
543,74680,NT,West Daly
544,79399,NT,Unincorporated NT
545,89399,ACT,Unincorporated ACT


In [911]:
# create date_dimension table
date_dim = df[['year', 'month']].drop_duplicates()
date_dim['dateID'] = df['year'].astype(str) + df['month'].astype(str)
date_dim = date_dim[['dateID', 'year', 'month']]
date_dim

# check if there are any NaN values in dateID
print(date_dim['dateID'].isna().sum())

0


In [912]:
# Create rush_dim table
rush_dim = df[['time_cat']].drop_duplicates()
rush_dim['rushID'] = range(1, len(rush_dim) + 1)
rush_dim = rush_dim[['rushID', 'time_cat']]
rush_dim.dropna(subset=['time_cat'], inplace=True)
rush_dim

,rushID,time_cat
0,1,Not Rush
7,2,Rush


In [913]:
# Create age_dim table
age_dim = df[['age_group']].drop_duplicates()

# sort by age_group
age_dim = age_dim.sort_values(by='age_group')

age_dim['ageID'] = range(1, len(age_dim) + 1)
age_dim = age_dim[['ageID', 'age_group']]
age_dim.dropna(subset=['age_group'], inplace=True)
age_dim


,ageID,age_group
20,1,0_to_16
1,2,17_to_25
2,3,26_to_39
4,4,40_to_64
0,5,65_to_74
18,6,75_or_older


In [914]:
# create daytime dimension table
daytime_dim = df[['time_of_day']].drop_duplicates()
daytime_dim['daytimeID'] = range(1, len(daytime_dim) + 1)
daytime_dim = daytime_dim[['daytimeID', 'time_of_day']]
daytime_dim.dropna(subset=['time_of_day'], inplace=True)
daytime_dim

,daytimeID,time_of_day
0,1,Night
1,2,Day


In [915]:
# create road_type dimension table
road_type_dim = df[['road_type']].drop_duplicates()
road_type_dim['road_typeID'] = range(1, len(road_type_dim) + 1)
road_type_dim = road_type_dim[['road_typeID', 'road_type']]
road_type_dim.dropna(subset=['road_type'], inplace=True)
road_type_dim

,road_typeID,road_type
0,1,Arterial Road
1,2,Local Road
3,3,National or State Highway
6,5,Sub-arterial Road
17,6,Collector Road
154,7,Pedestrian Thoroughfare
283,8,Access road
970,9,Busway


In [916]:
# create speed_limit dimension table
speed_limit_dim = df[['speed_limit']].drop_duplicates().dropna()
speed_limit_dim['speed_limitID'] = range(1, len(speed_limit_dim) + 1)
speed_limit_dim = speed_limit_dim[['speed_limitID', 'speed_limit']]
speed_limit_dim.dropna(subset=['speed_limit'], inplace=True)
speed_limit_dim

,speed_limitID,speed_limit
0,1,Very High
1,2,High
2,3,Med
25,4,Low


In [917]:
# create holiday dimension table
holiday_dim = df[['holiday']].drop_duplicates().sort_values(by='holiday')
holiday_dim['holidayID'] = range(1, len(holiday_dim) + 1)
holiday_dim = holiday_dim[['holidayID', 'holiday']]
holiday_dim.dropna(subset=['holiday'], inplace=True)
holiday_dim

,holidayID,holiday
0,1,Christmas
892,2,Easter
1,3,NoHoliday


In [918]:
# create vehicle_type dimension table
vehicle_type_dim = df[['vehicle_type_involved']].drop_duplicates().dropna()
vehicle_type_dim['vehicle_typeID'] = range(1, len(vehicle_type_dim) + 1)
vehicle_type_dim = vehicle_type_dim[['vehicle_typeID', 'vehicle_type_involved']]
vehicle_type_dim.dropna(subset=['vehicle_type_involved'], inplace=True)
vehicle_type_dim

,vehicle_typeID,vehicle_type_involved
0,1,No Heavy Vehicle Involved
13,2,Heavy Vehicle Involved


In [919]:
# save all dimension tables to csv with loop
dimension_tables = {
    "location_dim": location_dim,
    "date_dim": date_dim,
    "rush_dim": rush_dim,
    "age_dim": age_dim,
    "daytime_dim": daytime_dim,
    "road_type_dim": road_type_dim,
    "speed_limit_dim": speed_limit_dim,
    "holiday_dim": holiday_dim,
    "vehicle_type_dim": vehicle_type_dim
}
# Save each DataFrame to a CSV file
for name, table in dimension_tables.items():
    output_file_path = f"data/{name}.csv"
    table.to_csv(output_file_path, index=False)
    print(f"{name} data saved to {output_file_path}")

location_dim data saved to data/location_dim.csv
date_dim data saved to data/date_dim.csv
rush_dim data saved to data/rush_dim.csv
age_dim data saved to data/age_dim.csv
daytime_dim data saved to data/daytime_dim.csv
road_type_dim data saved to data/road_type_dim.csv
speed_limit_dim data saved to data/speed_limit_dim.csv
holiday_dim data saved to data/holiday_dim.csv
vehicle_type_dim data saved to data/vehicle_type_dim.csv


In [920]:
# Save the cleaned DataFrame for Algorithm Rule Mining
df_cleaned = df.copy()

In [921]:
fatalities_df = df.copy()
# fatalities_df = fatalities_df.merge(location_dim, on=['state', 'lga_name'], how='left')
fatalities_df = fatalities_df.merge(date_dim, on=['year', 'month'], how='left')
fatalities_df = fatalities_df.merge(rush_dim, on=['time_cat'], how='left')
fatalities_df = fatalities_df.merge(age_dim, on=['age_group'], how='left')
fatalities_df = fatalities_df.merge(daytime_dim, on=['time_of_day'], how='left')
fatalities_df = fatalities_df.merge(road_type_dim, on=['road_type'], how='left')
fatalities_df = fatalities_df.merge(speed_limit_dim, on=['speed_limit'], how='left')
fatalities_df = fatalities_df.merge(holiday_dim, on=['holiday'], how='left')
fatalities_df = fatalities_df.merge(vehicle_type_dim, on=['vehicle_type_involved'], how='left')

In [922]:
fatalities_df.columns

Index(['crash_id', 'victim_number', 'state', 'month', 'year', 'dayweek',
       'time', 'crash_type', 'bus_involvement',
       'heavy_rigid_truck_involvement', 'articulated_truck_involvement',
       'speed_limit', 'lga_name', 'road_type', 'christmas_period',
       'easter_period', 'age_group', 'time_of_day', 'time_cat', 'holiday',
       'vehicle_type_involved', 'lga_code', 'dateID', 'rushID', 'ageID',
       'daytimeID', 'road_typeID', 'speed_limitID', 'holidayID',
       'vehicle_typeID'],
      dtype='object')

In [923]:
# drop unnecessary columns
fatalities_df = fatalities_df[['crash_id', 'victim_number', 'lga_code', 'dateID', 'rushID', 'ageID', 'daytimeID', 'road_typeID', 'speed_limitID', 'holidayID', 'vehicle_typeID']]
# Convert all values to Int64 ingnoring NaN
fatalities_df = fatalities_df.astype(pd.Int64Dtype())
fatalities_df

,crash_id,victim_number,lga_code,dateID,rushID,ageID,daytimeID,road_typeID,speed_limitID,holidayID,vehicle_typeID
0,20241115,1,17750,202412,1,5,1,1,1,1,1
1,20241125,1,13800,202412,1,2,2,2,2,3,1
2,20246013,1,64610,202412,1,3,2,2,3,1,1
3,20241002,1,10180,202412,1,3,2,3,1,3,1
4,20242261,1,<NA>,202412,1,4,2,<NA>,<NA>,3,<NA>
...,...,...,...,...,...,...,...,...,...,...,...
56869,19896006,3,<NA>,19891,1,1,1,<NA>,1,3,2
56870,19896006,4,<NA>,19891,1,1,1,<NA>,1,3,2
56871,19896006,5,<NA>,19891,1,2,1,<NA>,1,3,2
56872,19896006,6,<NA>,19891,1,1,1,<NA>,1,3,2


In [924]:
fatalities_df.columns

Index(['crash_id', 'victim_number', 'lga_code', 'dateID', 'rushID', 'ageID',
       'daytimeID', 'road_typeID', 'speed_limitID', 'holidayID',
       'vehicle_typeID'],
      dtype='object')

In [925]:
population_df = population_long.copy()
population_df = population_df[['lga_code', 'year', 'population']]
population_df

,lga_code,year,population
0,10050,2001,45265
1,10180,2001,27906
2,10250,2001,37856
3,10300,2001,2751
4,10470,2001,35504
...,...,...,...
12576,74660,2023,7407
12577,74680,2023,3426
12578,79399,2023,7713
12579,89399,2023,466566


In [926]:
# save all fact tables to csv with loop
fact_tables = {
    "fatalities_fact": fatalities_df,
    "population_fact": population_df
}
# Save each DataFrame to a CSV file
for name, table in fact_tables.items():
    output_file_path = f"data/{name}.csv"
    table.to_csv(output_file_path, index=False)
    print(f"{name} data saved to {output_file_path}")

fatalities_fact data saved to data/fatalities_fact.csv
population_fact data saved to data/population_fact.csv


In [927]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded_array = encoder.fit_transform(df_cleaned[['state','month','year','speed_limit','lga_name','road_type','holiday','age_group','time_of_day','time_cat','vehicle_type_involved']])

encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(['state','month','year','speed_limit','lga_name','road_type','holiday','age_group','time_of_day','time_cat','vehicle_type_involved']))
print(encoded_df)

       state_ACT  state_NSW  state_NT  state_Qld  state_SA  state_Tas  \
0            0.0        1.0       0.0        0.0       0.0        0.0   
1            0.0        1.0       0.0        0.0       0.0        0.0   
2            0.0        0.0       0.0        0.0       0.0        1.0   
3            0.0        1.0       0.0        0.0       0.0        0.0   
4            0.0        0.0       0.0        0.0       0.0        0.0   
...          ...        ...       ...        ...       ...        ...   
56869        0.0        0.0       0.0        0.0       0.0        1.0   
56870        0.0        0.0       0.0        0.0       0.0        1.0   
56871        0.0        0.0       0.0        0.0       0.0        1.0   
56872        0.0        0.0       0.0        0.0       0.0        1.0   
56873        0.0        0.0       0.0        0.0       0.0        0.0   

       state_Vic  state_WA  month_1  month_2  ...  age_group_nan  \
0            0.0       0.0      0.0      0.0  ...      

In [929]:
!pip install mlxtend
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.frequent_patterns import fpgrowth, association_rules

  Using cached mlxtend-0.23.4-py3-none-any.whl.metadata (7.3 kB)
Using cached mlxtend-0.23.4-py3-none-any.whl (1.4 MB)


In [930]:
min_support = 0.05  


frequent_itemsets_ap = apriori(encoded_df, min_support=min_support, use_colnames=True)

print("Combos")
print(frequent_itemsets_ap.head())


min_confidence = 0.3
rules = association_rules(frequent_itemsets_ap, metric="confidence", min_threshold=min_confidence)

print("\n(Apriori):")
print(rules.head())

/Users/Kirill/miniconda3/envs/cits5508/lib/python3.11/site-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


Combos
    support     itemsets
0  0.304638  (state_NSW)
1  0.201146  (state_Qld)
2  0.085329   (state_SA)
3  0.218659  (state_Vic)
4  0.120301   (state_WA)

(Apriori):
          antecedents              consequents  antecedent support  \
0         (state_NSW)       (speed_limit_High)            0.304638   
1  (speed_limit_High)              (state_NSW)            0.421985   
2         (state_NSW)  (speed_limit_Very High)            0.304638   
3         (state_NSW)           (lga_name_nan)            0.304638   
4      (lga_name_nan)              (state_NSW)            0.806150   

   consequent support   support  confidence      lift  representativity  \
0            0.421985  0.137391    0.450998  1.068754               1.0   
1            0.304638  0.137391    0.325583  1.068754               1.0   
2            0.480518  0.138429    0.454404  0.945653               1.0   
3            0.806150  0.245930    0.807284  1.001406               1.0   
4            0.304638  0.245930    

In [931]:
min_support = 0.05
frequent_itemsets_ap = apriori(encoded_df, min_support=min_support, use_colnames=True, low_memory=True)

print("Combos:")
print(frequent_itemsets_ap.head())

min_confidence = 0.6
rules = association_rules(frequent_itemsets_ap, metric="confidence", min_threshold=min_confidence)


filtered_rules = rules[(rules['lift'] > 1.2) & (rules['leverage'] > 0) & (~rules['antecedents'].apply(lambda x: any('_nan' in s for s in x))) &
    (~rules['consequents'].apply(lambda x: any('_nan' in s for s in x)))]

print("\n(Apriori) All rules:")
print(rules.head())

print("\n(Apriori) Filtered rules (lift>1.4, leverage>0):")
print(filtered_rules.head())

/Users/Kirill/miniconda3/envs/cits5508/lib/python3.11/site-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


Combos:
    support     itemsets
0  0.304638  (state_NSW)
1  0.201146  (state_Qld)
2  0.085329   (state_SA)
3  0.218659  (state_Vic)
4  0.120301   (state_WA)

(Apriori) All rules:
   antecedents          consequents  antecedent support  consequent support  \
0  (state_NSW)       (lga_name_nan)            0.304638            0.806150   
1  (state_NSW)      (road_type_nan)            0.304638            0.811197   
2  (state_NSW)  (holiday_NoHoliday)            0.304638            0.962760   
3  (state_NSW)  (time_cat_Not Rush)            0.304638            0.847206   
4  (state_Qld)       (lga_name_nan)            0.201146            0.806150   

    support  confidence      lift  representativity  leverage  conviction  \
0  0.245930    0.807284  1.001406               1.0  0.000345    1.005881   
1  0.245947    0.807342  0.995248               1.0 -0.001174    0.979990   
2  0.293667    0.963985  1.001272               1.0  0.000373    1.034013   
3  0.261244    0.857555  1.012215    

In [932]:
output_file_path_2 = "algo.xlsx"
filtered_rules.to_excel(output_file_path_2, index=False)
print(f"Cleaned data saved to {output_file_path_2}")

Cleaned data saved to algo.xlsx


In [933]:
min_support = 0.05 


frequent_itemsets_fp = fpgrowth(encoded_df, min_support=min_support, use_colnames=True)

print("Combos")
print(frequent_itemsets_fp.head())


min_confidence = 0.6
rules_fp = association_rules(frequent_itemsets_fp, metric="confidence", min_threshold=min_confidence)


filtered_rules_2 = rules_fp[(rules_fp['lift'] > 1.4) & (rules_fp['leverage'] > 0) & (~rules['antecedents'].apply(lambda x: any('_nan' in s for s in x))) &
    (~rules['consequents'].apply(lambda x: any('_nan' in s for s in x)))]

print("\n(FP-Growth):")
print(rules_fp.head())

print("\n(FP-Growth) Filtered rules (lift>1.4, leverage>0):")
print(filtered_rules_2.head())

/Users/Kirill/miniconda3/envs/cits5508/lib/python3.11/site-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


Combos
    support                                           itemsets
0  0.847206                                (time_cat_Not Rush)
1  0.536168  (vehicle_type_involved_No Heavy Vehicle Involved)
2  0.480518                            (speed_limit_Very High)
3  0.429669                                (time_of_day_Night)
4  0.304638                                        (state_NSW)

(FP-Growth):
                                         antecedents          consequents  \
0                                (time_cat_Not Rush)  (holiday_NoHoliday)   
1                                (holiday_NoHoliday)  (time_cat_Not Rush)   
2  (vehicle_type_involved_No Heavy Vehicle Involved)  (time_cat_Not Rush)   
3  (vehicle_type_involved_No Heavy Vehicle Involved)  (holiday_NoHoliday)   
4  (vehicle_type_involved_No Heavy Vehicle Involved)      (road_type_nan)   

   antecedent support  consequent support   support  confidence      lift  \
0            0.847206            0.962760  0.810036    0.9561

In [934]:
output_file_path_2 = "algo_2.xlsx"
filtered_rules_2.to_excel(output_file_path_2, index=False)
print(f"Cleaned data saved to {output_file_path_2}")

Cleaned data saved to algo_2.xlsx
